# Clean Rel-Stack PRMP Benchmark with Learned Embeddings

**Predictive Residual Message Passing (PRMP)** on heterogeneous relational graphs.

This notebook benchmarks 3 GNN variants on a synthetic heterogeneous graph built from StackOverflow-like relational data:
- **Standard**: Mean-aggregation message passing (SAGEConv)
- **PRMP**: Parent predicts child features, aggregates *residuals* instead of raw features
- **Wide**: Standard with extra parameters to match PRMP parameter count

**Tasks**: post-votes regression (MAE) and user-engagement classification (AUROC).
All GNN layers are implemented in pure PyTorch (no torch-geometric).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All needed packages are pre-installed on Colab; install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')
    _pip('torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cpu')

In [ ]:
import copy
import gc
import itertools
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import RidgeCV
from sklearn.metrics import roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print(f"PyTorch {torch.__version__}, NumPy {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-b2d5b0-predictive-residual-message-passing-filt/main/experiment_iter6_clean_rel_stack/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")
print(f"Dataset: {data['metadata']['dataset']}")
print(f"Tables: {data['metadata']['num_tables']}, FK links: {data['metadata']['num_fk_links']}")
print(f"\nFull-scale results summary:")
for task, variants in data['metadata']['results_summary'].items():
    print(f"  {task}:")
    for v in ['Standard', 'PRMP', 'Wide']:
        if v in variants:
            vr = variants[v]
            metric_key = [k for k in vr if k.startswith('mean_')][0]
            print(f"    {v}: {metric_key}={vr[metric_key]:.6f}")

In [ ]:
# ============ DEMO CONFIG ============
# Minimum values for quick demo run. Original full-scale values in comments.
MAX_ROWS = 500        # Original: 100_000
EMBED_DIM = 32        # Original: 128
HIDDEN_DIM = 32       # Original: 128
NUM_LAYERS = 2        # Original: 2
DROPOUT = 0.3         # Original: 0.3
LR = 0.001            # Original: 0.001
WEIGHT_DECAY = 1e-5   # Original: 1e-5
MAX_EPOCHS = 20       # Original: 200
PATIENCE = 5          # Original: 20
SEEDS = [42]          # Original: [42, 123, 456]
# =====================================
print(f"Demo config: MAX_ROWS={MAX_ROWS}, EMBED_DIM={EMBED_DIM}, HIDDEN_DIM={HIDDEN_DIM}")
print(f"MAX_EPOCHS={MAX_EPOCHS}, PATIENCE={PATIENCE}, SEEDS={SEEDS}")

## Phase 1: Graph Construction

Build a heterogeneous graph from StackOverflow relational data statistics.
Uses known table sizes and FK cardinality distributions to create a realistic graph topology.
All node features come from `nn.Embedding` (defined in the model), so only
topology and task labels are needed here.

In [ ]:
# Table sizes from the full StackOverflow rel-stack dataset
TABLE_SIZES_FULL = {
    "users": 255_360, "posts": 333_893, "comments": 623_967,
    "badges": 463_463, "postLinks": 77_337, "postHistory": 1_175_368,
    "votes": 1_317_876,
}

FK_LINKS_DEF = [
    ("badges", "UserId", "users"), ("comments", "PostId", "posts"),
    ("comments", "UserId", "users"), ("postLinks", "PostId", "posts"),
    ("postLinks", "RelatedPostId", "posts"), ("postHistory", "PostId", "posts"),
    ("postHistory", "UserId", "users"), ("votes", "PostId", "posts"),
    ("votes", "UserId", "users"), ("posts", "OwnerUserId", "users"),
    ("posts", "ParentId", "posts"),
]

FK_COVERAGE = {
    "comments__UserId__users": 0.208, "comments__PostId__posts": 0.562,
    "badges__UserId__users": 0.555, "postLinks__PostId__posts": 0.113,
    "postLinks__RelatedPostId__posts": 0.074, "postHistory__PostId__posts": 1.000,
    "postHistory__UserId__users": 0.333, "votes__PostId__posts": 0.829,
    "votes__UserId__users": 0.010, "posts__OwnerUserId__users": 0.330,
    "posts__ParentId__posts": 0.346,
}

def build_graph(max_rows, seed=42):
    rng = np.random.RandomState(seed)
    num_nodes = {t: min(s, max_rows) for t, s in TABLE_SIZES_FULL.items()}
    print("Node counts:", {t: f"{n:,}" for t, n in num_nodes.items()})

    edge_data = {}
    fwd_edge_keys = set()
    for child_table, fkey_col, parent_table in FK_LINKS_DEF:
        link_id = f"{child_table}__{fkey_col}__{parent_table}"
        n_child, n_parent = num_nodes[child_table], num_nodes[parent_table]
        coverage = FK_COVERAGE.get(link_id, 0.5)
        if child_table == parent_table:
            n_edges = int(n_child * min(coverage * 1.5, 0.8))
        else:
            n_edges = int(n_child * min(coverage * 2.0, 0.95))
        n_edges = max(n_edges, min(1000, n_child))
        n_edges = min(n_edges, n_child)

        child_ids = rng.choice(n_child, size=n_edges, replace=False)
        weights = 1.0 / np.power(np.arange(1, n_parent + 1, dtype=np.float64), 0.7)
        weights /= weights.sum()
        parent_ids = rng.choice(n_parent, size=n_edges, p=weights)

        fwd_key = f"fwd_{link_id}"
        edge_data[fwd_key] = {
            "src_type": child_table, "dst_type": parent_table,
            "src_idx": torch.tensor(child_ids, dtype=torch.long),
            "dst_idx": torch.tensor(parent_ids, dtype=torch.long),
        }
        fwd_edge_keys.add(fwd_key)
        rev_key = f"rev_{link_id}"
        edge_data[rev_key] = {
            "src_type": parent_table, "dst_type": child_table,
            "src_idx": torch.tensor(parent_ids, dtype=torch.long),
            "dst_idx": torch.tensor(child_ids, dtype=torch.long),
        }

    # Task 1: post-votes regression — target = log(1 + incoming vote count)
    vote_counts = np.zeros(num_nodes["posts"], dtype=np.float32)
    if "fwd_votes__PostId__posts" in edge_data:
        dst = edge_data["fwd_votes__PostId__posts"]["dst_idx"].numpy()
        np.add.at(vote_counts, dst, 1.0)
    y_reg = np.log1p(vote_counts)

    # Task 2: user-engagement classification — target = above-median activity
    activity = np.zeros(num_nodes["users"], dtype=np.float32)
    for ed in edge_data.values():
        if ed["dst_type"] == "users":
            np.add.at(activity, ed["dst_idx"].numpy(), 1.0)
    threshold = max(np.median(activity[activity > 0]), 1.0)
    y_cls = (activity >= threshold).astype(np.float32)
    if y_cls.mean() < 0.1 or y_cls.mean() > 0.9:
        y_cls = (activity > np.percentile(activity, 50)).astype(np.float32)

    n_posts, n_users = num_nodes["posts"], num_nodes["users"]
    perm_p, perm_u = rng.permutation(n_posts), rng.permutation(n_users)
    task_info = {
        "post-votes": {
            "type": "regression", "metric": "MAE", "loss": "L1",
            "target_node_type": "posts",
            "y": torch.tensor(y_reg, dtype=torch.float32),
            "train_mask": torch.tensor(perm_p[:int(0.8*n_posts)], dtype=torch.long),
            "val_mask": torch.tensor(perm_p[int(0.8*n_posts):int(0.9*n_posts)], dtype=torch.long),
            "test_mask": torch.tensor(perm_p[int(0.9*n_posts):], dtype=torch.long),
        },
        "user-engagement": {
            "type": "classification", "metric": "AUROC", "loss": "BCE",
            "target_node_type": "users",
            "y": torch.tensor(y_cls, dtype=torch.float32),
            "train_mask": torch.tensor(perm_u[:int(0.8*n_users)], dtype=torch.long),
            "val_mask": torch.tensor(perm_u[int(0.8*n_users):int(0.9*n_users)], dtype=torch.long),
            "test_mask": torch.tensor(perm_u[int(0.9*n_users):], dtype=torch.long),
        },
    }
    print(f"post-votes target: mean={y_reg.mean():.3f}, std={y_reg.std():.3f}")
    print(f"user-engagement: pos_rate={y_cls.mean():.3f}")
    return {"num_nodes": num_nodes, "edge_data": edge_data,
            "fwd_edge_keys": fwd_edge_keys, "task_info": task_info}

graph = build_graph(MAX_ROWS)

# Move to device
for ed in graph["edge_data"].values():
    ed["src_idx"] = ed["src_idx"].to(DEVICE)
    ed["dst_idx"] = ed["dst_idx"].to(DEVICE)
for ti in graph["task_info"].values():
    ti["y"] = ti["y"].to(DEVICE)
    ti["train_mask"] = ti["train_mask"].to(DEVICE)
    ti["val_mask"] = ti["val_mask"].to(DEVICE)
    ti["test_mask"] = ti["test_mask"].to(DEVICE)
print(f"\nGraph built and moved to {DEVICE}")

## Phase 2: Pure PyTorch GNN Layers

The key innovation is **Predictive Residual Message Passing (PRMP)**:
- **Standard GNN**: aggregate neighbor features directly via mean pooling
- **PRMP**: parent node *predicts* child features via a small MLP, then aggregates the *residuals* (prediction errors)

This focuses the message on *surprising* information that the parent couldn't predict,
filtering out redundant structural information.

In [ ]:
def scatter_mean(src, index, dim_size):
    """Scatter mean: aggregate src by index."""
    out = torch.zeros(dim_size, src.size(1), device=src.device, dtype=src.dtype)
    count = torch.zeros(dim_size, 1, device=src.device, dtype=src.dtype)
    out.scatter_add_(0, index.unsqueeze(1).expand_as(src), src)
    ones = torch.ones(src.size(0), 1, device=src.device, dtype=src.dtype)
    count.scatter_add_(0, index.unsqueeze(1), ones)
    return out / count.clamp(min=1)


class BipartiteSAGEConv(nn.Module):
    """Standard SAGEConv: mean-aggregate src features to dst nodes."""
    def __init__(self, src_dim, dst_dim, out_dim):
        super().__init__()
        self.lin_neigh = nn.Linear(src_dim, out_dim)
        self.lin_self = nn.Linear(dst_dim, out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        agg = scatter_mean(x_src[edge_src], edge_dst, num_dst)
        return self.lin_neigh(agg) + self.lin_self(x_dst)


class PRMPSAGEConv(nn.Module):
    """PRMP: parent predicts child features, aggregates residuals."""
    def __init__(self, src_dim, dst_dim, out_dim):
        super().__init__()
        self.pred_mlp = nn.Sequential(
            nn.Linear(dst_dim, src_dim), nn.ReLU(), nn.Linear(src_dim, src_dim))
        # Initialize last layer near zero so residuals start as raw features
        nn.init.zeros_(self.pred_mlp[2].weight)
        nn.init.zeros_(self.pred_mlp[2].bias)
        self.lin_neigh = nn.Linear(src_dim, out_dim)
        self.lin_self = nn.Linear(dst_dim, out_dim)

    def forward(self, x_src, x_dst, edge_src, edge_dst, num_dst):
        pred_src = self.pred_mlp(x_dst)                     # parent predicts child
        residual = x_src[edge_src] - pred_src[edge_dst]     # THE CORE PRMP OPERATION
        agg = scatter_mean(residual, edge_dst, num_dst)
        return self.lin_neigh(agg) + self.lin_self(x_dst)

print("GNN layers defined: BipartiteSAGEConv (standard), PRMPSAGEConv (residual)")

## Phase 3: Heterogeneous GNN Model

A 2-layer heterogeneous GNN with `nn.Embedding` for all 7 node types.
For the **PRMP** variant, forward FK edges use `PRMPSAGEConv` while
reverse edges use standard `BipartiteSAGEConv`.

In [ ]:
class HeteroGNNLayer(nn.Module):
    """One layer of heterogeneous message passing across all edge types."""
    def __init__(self, conv_dict):
        super().__init__()
        self.convs = conv_dict

    def forward(self, x_dict, edge_data, num_nodes):
        out_dict = {}
        for etype, conv in self.convs.items():
            ed = edge_data[etype]
            result = conv(x_dict[ed["src_type"]], x_dict[ed["dst_type"]],
                          ed["src_idx"], ed["dst_idx"], num_nodes[ed["dst_type"]])
            if ed["dst_type"] not in out_dict:
                out_dict[ed["dst_type"]] = result
            else:
                out_dict[ed["dst_type"]] = out_dict[ed["dst_type"]] + result
        for ntype in x_dict:
            if ntype not in out_dict:
                out_dict[ntype] = x_dict[ntype]
        return out_dict


class HeteroGNNModel(nn.Module):
    """Heterogeneous GNN with nn.Embedding for all node types."""
    def __init__(self, num_nodes_dict, hidden_dim, edge_type_names, edge_type_info,
                 target_node_type, variant="Standard", fwd_edge_keys=None):
        super().__init__()
        self.variant = variant
        self.hidden_dim = hidden_dim
        self.target_node_type = target_node_type
        self.embeddings = nn.ModuleDict({
            t: nn.Embedding(n, EMBED_DIM) for t, n in num_nodes_dict.items()
        })
        if hidden_dim != EMBED_DIM:
            self.input_proj = nn.ModuleDict({
                t: nn.Linear(EMBED_DIM, hidden_dim) for t in num_nodes_dict
            })
        else:
            self.input_proj = None

        def _build_layer():
            conv_dict = {}
            for etype in edge_type_names:
                if variant == "PRMP" and fwd_edge_keys and etype in fwd_edge_keys:
                    conv_dict[etype] = PRMPSAGEConv(hidden_dim, hidden_dim, hidden_dim)
                else:
                    conv_dict[etype] = BipartiteSAGEConv(hidden_dim, hidden_dim, hidden_dim)
            return HeteroGNNLayer(nn.ModuleDict(conv_dict))

        self.conv1 = _build_layer()
        self.conv2 = _build_layer()
        self.head = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, edge_data, num_nodes):
        x_dict = {t: emb.weight for t, emb in self.embeddings.items()}
        if self.input_proj is not None:
            x_dict = {t: F.relu(self.input_proj[t](x)) for t, x in x_dict.items()}
        else:
            x_dict = {t: F.relu(x) for t, x in x_dict.items()}
        out = self.conv1(x_dict, edge_data, num_nodes)
        x_dict = {t: F.relu(self.dropout(v)) for t, v in out.items()}
        out = self.conv2(x_dict, edge_data, num_nodes)
        x_dict = {t: F.relu(v) for t, v in out.items()}
        return self.head(x_dict[self.target_node_type]).squeeze(-1)

    def get_embeddings(self):
        """Extract learned embeddings (detached, on CPU)."""
        return {t: emb.weight.detach().cpu().numpy()
                for t, emb in self.embeddings.items()}

print("HeteroGNNModel defined")

## Phase 4: Training All Variants

Train **Standard**, **PRMP**, and **Wide** (parameter-matched) variants on both tasks.
The Wide variant uses a larger hidden dimension so its total parameter count matches PRMP,
enabling a fair comparison of whether PRMP's benefit comes from the residual mechanism
or just from having more parameters.

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def compute_wide_dim(num_nodes_dict, edge_type_names):
    """Compute hidden_dim for Wide variant to match PRMP param count."""
    n_et = len(edge_type_names)
    n_fwd = n_et // 2
    n_tab = len(num_nodes_dict)
    h = HIDDEN_DIM
    std_conv = n_fwd * (2 * h * h + 2 * h)
    prmp_conv = n_fwd * (4 * h * h + 4 * h)
    prmp_total = 2 * (std_conv + prmp_conv) + h + 1
    a = 4 * n_et
    b = n_tab * (EMBED_DIM + 1) + 4 * n_et + 1
    c = 1 - prmp_total
    disc = b * b - 4 * a * c
    if disc < 0:
        return HIDDEN_DIM + 20
    return max(int((-b + math.sqrt(disc)) / (2 * a)), HIDDEN_DIM + 1)

def train_one_run(model, graph, task_name, seed, device):
    """Train one model for one task/seed combination."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    ti = graph["task_info"][task_name]
    y, tr_m, va_m, te_m = ti["y"], ti["train_mask"], ti["val_mask"], ti["test_mask"]
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.L1Loss() if ti["loss"] == "L1" else nn.BCEWithLogitsLoss()
    best_val = float("inf") if ti["type"] == "regression" else float("-inf")
    patience_ctr, best_state, best_ep = 0, None, 0

    for epoch in range(MAX_EPOCHS):
        model.train()
        opt.zero_grad()
        pred = model(graph["edge_data"], graph["num_nodes"])
        loss = loss_fn(pred[tr_m], y[tr_m])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        model.eval()
        with torch.no_grad():
            pred = model(graph["edge_data"], graph["num_nodes"])
            if ti["type"] == "regression":
                vm = F.l1_loss(pred[va_m], y[va_m]).item()
                improved = vm < best_val
            else:
                try:
                    vm = roc_auc_score(y[va_m].cpu().numpy(), pred[va_m].cpu().numpy())
                except ValueError:
                    vm = 0.5
                improved = vm > best_val
        if improved:
            best_val = vm
            patience_ctr = 0
            best_state = copy.deepcopy(model.state_dict())
            best_ep = epoch
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(graph["edge_data"], graph["num_nodes"])
        tp = pred[te_m].cpu().numpy()
        tt = y[te_m].cpu().numpy()

    if ti["type"] == "regression":
        mae = float(np.mean(np.abs(tp - tt)))
        ss_res = float(np.sum((tt - tp) ** 2))
        ss_tot = float(np.sum((tt - tt.mean()) ** 2))
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        return {"mae": round(mae, 6), "r2": round(r2, 6),
                "best_epoch": best_ep, "seed": seed}
    else:
        try:
            auroc = float(roc_auc_score(tt, tp))
        except ValueError:
            auroc = 0.5
        return {"auroc": round(auroc, 6), "best_epoch": best_ep, "seed": seed}

# --- Run all training ---
edge_type_names = list(graph["edge_data"].keys())
edge_type_info = {k: {"src_type": v["src_type"], "dst_type": v["dst_type"]}
                  for k, v in graph["edge_data"].items()}
wide_dim = compute_wide_dim(graph["num_nodes"], edge_type_names)

variants_cfg = {
    "Standard": {"hidden_dim": HIDDEN_DIM, "variant": "Standard"},
    "PRMP": {"hidden_dim": HIDDEN_DIM, "variant": "PRMP"},
    "Wide": {"hidden_dim": wide_dim, "variant": "Standard"},
}

# Show parameter counts
for vname, vcfg in variants_cfg.items():
    m = HeteroGNNModel(graph["num_nodes"], vcfg["hidden_dim"], edge_type_names,
                       edge_type_info, "posts", vcfg["variant"], graph["fwd_edge_keys"])
    print(f"  {vname}: {count_parameters(m):,} params (hidden_dim={vcfg['hidden_dim']})")
    del m

all_results = {}
all_models = {}
t_start = time.time()

for task_name in ["post-votes", "user-engagement"]:
    target_type = graph["task_info"][task_name]["target_node_type"]
    all_results[task_name] = {}
    for vname, vcfg in variants_cfg.items():
        all_results[task_name][vname] = []
        for seed in SEEDS:
            print(f"Training: {task_name} | {vname} | seed={seed} ...", end=" ")
            model = HeteroGNNModel(
                graph["num_nodes"], vcfg["hidden_dim"], edge_type_names,
                edge_type_info, target_type, vcfg["variant"],
                graph["fwd_edge_keys"]).to(DEVICE)
            result = train_one_run(model, graph, task_name, seed, DEVICE)
            all_results[task_name][vname].append(result)
            all_models[f"{task_name}__{vname}"] = model.cpu()
            print(result)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

print(f"\nAll training completed in {time.time() - t_start:.1f}s")

## Phase 5: Embedding R² Analysis

Measure how well parent embeddings predict child embeddings across each FK link
using Ridge regression. Higher R² means the GNN learned embeddings that
capture the relational structure.

In [ ]:
def compute_embedding_r2(all_models, graph):
    """Compute R^2 for predicting child embeddings from parent embeddings."""
    r2_results = {}
    for model_key, model in all_models.items():
        task_name, variant = model_key.split("__")
        emb_dict = model.get_embeddings()
        for child_table, fkey_col, parent_table in FK_LINKS_DEF:
            link_id = f"{child_table}__{fkey_col}__{parent_table}"
            fwd_key = f"fwd_{link_id}"
            if fwd_key not in graph["edge_data"]:
                continue
            ed = graph["edge_data"][fwd_key]
            si = ed["src_idx"].cpu().numpy()
            di = ed["dst_idx"].cpu().numpy()
            if len(si) > 5000:
                sel = np.random.RandomState(42).choice(len(si), 5000, replace=False)
                si, di = si[sel], di[sel]
            try:
                ridge = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0])
                ridge.fit(emb_dict[parent_table][di], emb_dict[child_table][si])
                r2 = float(ridge.score(emb_dict[parent_table][di], emb_dict[child_table][si]))
            except Exception:
                r2 = 0.0
            r2_results.setdefault(link_id, {}).setdefault(variant, {})[task_name] = round(r2, 6)
    return r2_results

embedding_r2 = compute_embedding_r2(all_models, graph)
print("Embedding R^2 per FK link (averaged across tasks):")
for link_id in sorted(embedding_r2.keys()):
    parts = []
    for v in ["Standard", "PRMP", "Wide"]:
        if v in embedding_r2[link_id]:
            vals = list(embedding_r2[link_id][v].values())
            parts.append(f"{v}={np.mean(vals):.4f}")
    print(f"  {link_id}: {', '.join(parts)}")

## Results & Visualization

Compare the demo (small-scale) results with the full-scale pre-computed results from the
complete experiment run (100K nodes, 200 epochs, 3 seeds).

In [ ]:
# --- Collect demo results ---
demo_summary = {}
for task_name, task_results in all_results.items():
    metric_key = "mae" if graph["task_info"][task_name]["type"] == "regression" else "auroc"
    demo_summary[task_name] = {}
    for vname, vresults in task_results.items():
        vals = [r[metric_key] for r in vresults]
        demo_summary[task_name][vname] = {"mean": np.mean(vals), "std": np.std(vals)}

# --- Collect full-scale results from loaded data ---
full_summary = {}
for task_name, task_variants in data["metadata"]["results_summary"].items():
    full_summary[task_name] = {}
    for vname in ["Standard", "PRMP", "Wide"]:
        if vname in task_variants:
            vr = task_variants[vname]
            mk = [k for k in vr if k.startswith("mean_")][0]
            sk = [k for k in vr if k.startswith("std_")][0]
            full_summary[task_name][vname] = {"mean": vr[mk], "std": vr[sk]}

# --- Print comparison table ---
print("=" * 75)
print(f"{'Task':<25} {'Variant':<10} {'Demo':>12} {'Full-scale':>12}")
print("=" * 75)
for task_name in ["post-votes", "user-engagement"]:
    metric = "MAE" if task_name == "post-votes" else "AUROC"
    for vname in ["Standard", "PRMP", "Wide"]:
        d = demo_summary[task_name][vname]
        f = full_summary[task_name][vname]
        label = f"{task_name} ({metric})"
        print(f"{label:<25} {vname:<10} {d['mean']:>12.4f} {f['mean']:>12.6f}")
    print("-" * 75)

# --- Plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
variants_list = ["Standard", "PRMP", "Wide"]
colors = {"Standard": "#4C72B0", "PRMP": "#C44E52", "Wide": "#55A868"}

# Plot 1: Post-votes MAE
ax = axes[0]
x = np.arange(len(variants_list))
w = 0.35
demo_vals = [demo_summary["post-votes"][v]["mean"] for v in variants_list]
full_vals = [full_summary["post-votes"][v]["mean"] for v in variants_list]
bars1 = ax.bar(x - w/2, demo_vals, w, label="Demo (small-scale)", color="#4C72B0", alpha=0.8)
bars2 = ax.bar(x + w/2, full_vals, w, label="Full-scale (100K nodes)", color="#DD8452", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(variants_list)
ax.set_ylabel("MAE (lower is better)")
ax.set_title("Post-Votes Regression")
ax.legend(fontsize=8)

# Plot 2: User-engagement AUROC
ax = axes[1]
demo_vals = [demo_summary["user-engagement"][v]["mean"] for v in variants_list]
full_vals = [full_summary["user-engagement"][v]["mean"] for v in variants_list]
ax.bar(x - w/2, demo_vals, w, label="Demo (small-scale)", color="#4C72B0", alpha=0.8)
ax.bar(x + w/2, full_vals, w, label="Full-scale (100K nodes)", color="#DD8452", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(variants_list)
ax.set_ylabel("AUROC (higher is better)")
ax.set_title("User-Engagement Classification")
ax.legend(fontsize=8)

# Plot 3: Full-scale Embedding R^2 per FK link
ax = axes[2]
full_r2 = data["metadata"]["embedding_r2_summary"]
fk_links = sorted(full_r2.keys())
fk_short = [fk.replace("__", ".").split(".")[0][:5] + ">" + fk.split("__")[2][:4]
            for fk in fk_links]
for i, v in enumerate(variants_list):
    vals = [full_r2[fk].get(f"{v}_mean", 0) or 0 for fk in fk_links]
    ax.bar(np.arange(len(fk_links)) + i * 0.25, vals, 0.25,
           label=v, color=colors[v], alpha=0.8)
ax.set_xticks(np.arange(len(fk_links)) + 0.25)
ax.set_xticklabels(fk_short, rotation=55, ha="right", fontsize=7)
ax.set_ylabel("Embedding R\u00b2")
ax.set_title("Embedding R\u00b2 per FK Link (Full-scale)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("results_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved to results_comparison.png")